# 🦺 PPE Compliance Violation Detection — YOLOv8 Training on Google Colab

> **Run this notebook top-to-bottom on Google Colab with a GPU runtime.**  
> Runtime → Change runtime type → **T4 GPU** (free) or A100 (Colab Pro)

| Step | What happens |
|------|--------------|
| 0 | Verify GPU & install deps |
| 1 | Clone project repo from GitHub |
| 2 | Download & split the PPE dataset |
| 3 | Explore class distribution |
| 4 | Train YOLOv8 |
| 5 | Plot training curves |
| 6 | Evaluate on test split |
| 7 | Visualise predictions |
| 8 | Export to ONNX |
| 9 | Download best weights |

---
## ✅ Step 0 — GPU Check & Install Dependencies

In [ ]:
# ── Verify GPU ─────────────────────────────────────────────
!nvidia-smi

In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
print('GPU name       :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None — switch runtime to GPU!')
print('PyTorch version:', torch.__version__)

In [ ]:
# ── Install Ultralytics (YOLOv8) ───────────────────────────
!pip install ultralytics -q

# Verify
from ultralytics import YOLO
import ultralytics
print('Ultralytics version:', ultralytics.__version__)

---
## 📂 Step 1 — Clone Project Repo from GitHub

In [ ]:
import os

PROJECT_REPO = 'https://github.com/Amitesh-Verma003/gril_assessment_3.git'
PROJECT_DIR  = '/content/gril_assessment_3'

if not os.path.exists(PROJECT_DIR):
    !git clone {PROJECT_REPO} {PROJECT_DIR}
else:
    print('Repo already cloned — pulling latest...')
    !git -C {PROJECT_DIR} pull

os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())
!ls -la

---
## 📥 Step 2 — Download & Prepare the PPE Dataset

In [ ]:
DATASET_REPO = 'https://github.com/Simuletic/cctv-ppe-compliance-detection-dataset.git'
RAW_DIR      = 'dataset/raw'

if not os.path.exists(RAW_DIR):
    print('Cloning PPE dataset (this may take a few minutes)...')
    !git clone {DATASET_REPO} {RAW_DIR}
else:
    print('Dataset already present — skipping clone.')

!echo "\nDataset folder contents:"
!ls dataset/raw/

In [ ]:
# ── Inspect raw dataset structure ──────────────────────────
!find dataset/raw -type f | head -30
!echo "---"
!find dataset/raw -name '*.txt' | wc -l
!find dataset/raw -name '*.jpg' -o -name '*.png' | wc -l

In [ ]:
# ── Split into train / val / test ──────────────────────────
# This script copies image+label pairs into dataset/images/ and dataset/labels/
!python scripts/prepare_dataset.py --val_ratio 0.15 --test_ratio 0.05

print('\nSplit complete:')
for split in ('train', 'val', 'test'):
    imgs = !find dataset/images/{split} -type f 2>/dev/null
    lbls = !find dataset/labels/{split} -name '*.txt' 2>/dev/null
    print(f'  {split:5s}: {len(imgs):5d} images | {len(lbls):5d} labels')

In [ ]:
# ── Confirm dataset YAML looks right ───────────────────────
!cat dataset/ppe_dataset.yaml

---
## 📊 Step 3 — Explore Dataset

In [ ]:
import cv2, random, matplotlib.pyplot as plt
from pathlib import Path

CLASS_NAMES = ['person','hardhat','hardhat_wrong','no_hardhat',
               'vest','vest_wrong','no_vest','gloves','no_gloves']
CMAP = plt.cm.tab10.colors

def show_samples(split='train', n=4, seed=7):
    imgs = list(Path(f'dataset/images/{split}').glob('*.*'))
    random.seed(seed)
    sample = random.sample(imgs, min(n, len(imgs)))
    fig, axes = plt.subplots(1, len(sample), figsize=(5*len(sample), 5))
    if len(sample) == 1: axes = [axes]
    for ax, ip in zip(axes, sample):
        img = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        lp = Path(f'dataset/labels/{split}') / (ip.stem + '.txt')
        if lp.exists():
            for line in lp.read_text().splitlines():
                cls, cx, cy, bw, bh = map(float, line.split())
                cls = int(cls)
                x1=int((cx-bw/2)*w); y1=int((cy-bh/2)*h)
                x2=int((cx+bw/2)*w); y2=int((cy+bh/2)*h)
                c = tuple(int(x*255) for x in CMAP[cls % 10])
                cv2.rectangle(img,(x1,y1),(x2,y2),c,2)
                cv2.putText(img,CLASS_NAMES[cls],(x1,max(y1-5,15)),
                            cv2.FONT_HERSHEY_SIMPLEX,0.5,c,1)
        ax.imshow(img); ax.axis('off')
        ax.set_title(ip.name, fontsize=7)
    plt.suptitle(f'Sample — {split} split', fontsize=12)
    plt.tight_layout(); plt.show()

show_samples('train', n=4)

In [ ]:
# ── Class distribution bar chart ───────────────────────────
import collections
counter = collections.Counter()
for lbl in Path('dataset/labels/train').glob('*.txt'):
    for line in lbl.read_text().splitlines():
        if line.strip():
            counter[CLASS_NAMES[int(line.split()[0])]] += 1

VIOLATION_CLS = {'hardhat_wrong','no_hardhat','vest_wrong','no_vest','no_gloves'}
colors = ['#e74c3c' if k in VIOLATION_CLS else '#2ecc71' for k in counter]

plt.figure(figsize=(11,4))
bars = plt.bar(counter.keys(), counter.values(), color=colors)
plt.title('Class Distribution — Training Set  (🔴 violation  🟢 compliant/anchor)')
plt.xlabel('Class'); plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
for b,v in zip(bars, counter.values()):
    plt.text(b.get_x()+b.get_width()/2, b.get_height()+8, str(v), ha='center', fontsize=9)
plt.tight_layout(); plt.show()

---
## 🏋️ Step 4 — Train YOLOv8

| Variant | Params | Colab GPU | Recommended when |
|---------|--------|-----------|------------------|
| `yolov8n` | 3.2M | T4 ✅ | fast test run |
| `yolov8s` | 11M | T4 ✅ | balanced |
| **`yolov8m`** | **26M** | **T4 ✅** | **best for assignment** |
| `yolov8l` | 44M | A100 | high accuracy |

> 💡 Change `MODEL_VARIANT` below to switch model size.

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# ══════════════════════════════════
#  ⚙️  CONFIGURE YOUR RUN HERE
# ══════════════════════════════════
MODEL_VARIANT = 'yolov8n'   # n | s | m | l
EPOCHS        = 50
BATCH_SIZE    = 16          # reduce to 8 if Out-of-Memory
IMG_SIZE      = 640
DEVICE        = 0           # 0 = first GPU
# ══════════════════════════════════

DATA_YAML = Path('dataset/ppe_dataset.yaml').resolve()
print('Dataset YAML:', DATA_YAML)

model = YOLO(f'{MODEL_VARIANT}.pt')   # downloads COCO pretrained weights

results = model.train(
    data          = str(DATA_YAML),
    epochs        = EPOCHS,
    batch         = BATCH_SIZE,
    imgsz         = IMG_SIZE,
    device        = DEVICE,
    project       = 'runs/train',
    name          = f'ppe_{MODEL_VARIANT}',
    exist_ok      = True,
    # ── Optimiser ──────────────────
    optimizer     = 'AdamW',
    lr0           = 1e-3,
    lrf           = 0.01,
    momentum      = 0.937,
    weight_decay  = 5e-4,
    warmup_epochs = 3,
    # ── Augmentation ───────────────
    augment       = True,
    mosaic        = 1.0,
    mixup         = 0.1,
    copy_paste    = 0.1,
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    degrees       = 10.0,
    translate     = 0.1,
    scale         = 0.5,
    fliplr        = 0.5,
    # ── Misc ───────────────────────
    patience      = 20,
    save_period   = 10,
    val           = True,
    plots         = True,
)

BEST_WEIGHTS = f'runs/train/ppe_{MODEL_VARIANT}/weights/best.pt'
print('\n✅ Training complete!')
print(f'   Best weights → {BEST_WEIGHTS}')

---
## 📈 Step 5 — Training Curves

In [ ]:
import pandas as pd
log_csv = Path(f'runs/train/ppe_{MODEL_VARIANT}/results.csv')
df = pd.read_csv(log_csv)
df.columns = df.columns.str.strip()
print(df.columns.tolist())

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
plot_cfg = [
    ('train/box_loss',       'val/box_loss',       'Box Loss'),
    ('train/cls_loss',       'val/cls_loss',       'Class Loss'),
    ('train/dfl_loss',       'val/dfl_loss',       'DFL Loss'),
    ('metrics/precision(B)', 'metrics/recall(B)',  'Precision vs Recall'),
    ('metrics/mAP50(B)',     None,                 'mAP@50'),
    ('metrics/mAP50-95(B)',  None,                 'mAP@50-95'),
]
for ax, (c1, c2, title) in zip(axes.flat, plot_cfg):
    if c1 in df: ax.plot(df[c1], label=c1.split('/')[-1])
    if c2 and c2 in df: ax.plot(df[c2], label=c2.split('/')[-1])
    ax.set_title(title); ax.legend(); ax.set_xlabel('Epoch')
plt.suptitle(f'Training Curves — ppe_{MODEL_VARIANT}', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# ── Show auto-generated Ultralytics plots ──────────────────
from IPython.display import Image, display
run_dir = Path(f'runs/train/ppe_{MODEL_VARIANT}')

for plot_name in ['results.png', 'confusion_matrix.png',
                  'PR_curve.png', 'F1_curve.png', 'labels.jpg']:
    p = run_dir / plot_name
    if p.exists():
        print(f'── {plot_name} ──')
        display(Image(str(p), width=700))

---
## 🧪 Step 6 — Evaluate on Test Set

In [ ]:
eval_model = YOLO(BEST_WEIGHTS)

metrics = eval_model.val(
    data     = str(DATA_YAML),
    split    = 'test',
    imgsz    = IMG_SIZE,
    device   = DEVICE,
    plots    = True,
    save_json= True,
)

print('\n' + '='*45)
print('       TEST SET EVALUATION RESULTS')
print('='*45)
print(f'  mAP@50     : {metrics.box.map50:.4f}')
print(f'  mAP@50-95  : {metrics.box.map:.4f}')
print(f'  Precision  : {metrics.box.mp:.4f}')
print(f'  Recall     : {metrics.box.mr:.4f}')
print('='*45)
print('\nPer-class AP50:')
names = eval_model.names
for i, ap in enumerate(metrics.box.ap50):
    print(f'  {names[i]:17s}: {ap:.4f}')

---
## 🖼️ Step 7 — Visualise Predictions with Violation Flags

In [ ]:
VIOLATION_MAP = {
    2: 'hardhat_wrong', 3: 'no_hardhat',
    5: 'vest_wrong',    6: 'no_vest',
    8: 'no_gloves'
}
COLORS = {
    0:(255,165,0),  1:(0,200,0),   2:(0,0,220),
    3:(220,0,0),    4:(0,200,0),   5:(0,0,220),
    6:(220,0,0),    7:(0,200,0),   8:(220,0,0),
}

def annotate(img_bgr, result):
    img = img_bgr.copy()
    violation = False
    for box, cls_id, conf in zip(
        result.boxes.xyxy.cpu().numpy(),
        result.boxes.cls.int().cpu().tolist(),
        result.boxes.conf.cpu().tolist()
    ):
        x1,y1,x2,y2 = map(int,box)
        c = COLORS.get(cls_id,(200,200,200))
        label = f'{CLASS_NAMES[cls_id]} {conf:.2f}'
        cv2.rectangle(img,(x1,y1),(x2,y2),c,2)
        (tw,th),_ = cv2.getTextSize(label,cv2.FONT_HERSHEY_SIMPLEX,0.5,1)
        cv2.rectangle(img,(x1,y1-th-6),(x1+tw+2,y1),c,-1)
        cv2.putText(img,label,(x1+1,y1-4),cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,255,255),1)
        if cls_id in VIOLATION_MAP: violation = True
    banner = '⚠ VIOLATION' if violation else '✓ COMPLIANT'
    bc = (0,0,180) if violation else (0,160,0)
    cv2.rectangle(img,(0,0),(img.shape[1],30),bc,-1)
    cv2.putText(img,banner,(6,21),cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)
    return img, violation

# ── Run on 6 random test images ───────────────────────────
test_imgs = random.sample(list(Path('dataset/images/test').glob('*.*')), 6)
preds     = eval_model.predict(test_imgs, conf=0.4, iou=0.45,
                                imgsz=IMG_SIZE, device=DEVICE, verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, r in zip(axes.flat, preds):
    raw  = cv2.imread(r.path)
    vis, viol = annotate(raw, r)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.axis('off')
    cls_ids = r.boxes.cls.int().cpu().tolist()
    names_v = list({VIOLATION_MAP[c] for c in cls_ids if c in VIOLATION_MAP})
    ax.set_title('\n'.join(names_v) if names_v else 'All OK', fontsize=8,
                 color='red' if names_v else 'green')
plt.suptitle('Test-set Predictions — PPE Violation Detection', fontsize=14)
plt.tight_layout(); plt.show()

---
## 📦 Step 8 — Export Model (ONNX)

In [ ]:
export_model = YOLO(BEST_WEIGHTS)
export_model.export(format='onnx', imgsz=IMG_SIZE, simplify=True)

onnx_path = BEST_WEIGHTS.replace('.pt', '.onnx')
print(f'ONNX model saved → {onnx_path}')

---
## 💾 Step 9 — Download Best Weights to Your PC

In [ ]:
from google.colab import files
import shutil, os

# ── Zip the entire run folder (weights + plots + metrics) ──
run_folder = f'runs/train/ppe_{MODEL_VARIANT}'
zip_name   = f'ppe_{MODEL_VARIANT}_results'
shutil.make_archive(zip_name, 'zip', run_folder)
print(f'Zipped → {zip_name}.zip  ({os.path.getsize(zip_name+".zip")/1e6:.1f} MB)')

# Download to browser
files.download(f'{zip_name}.zip')

In [ ]:
# ── Or just the best.pt weight file ───────────────────────
files.download(BEST_WEIGHTS)

---
## 🔁 (Optional) Save back to GitHub
Only run if you want to commit the trained weights / results back to your repo.

In [ ]:
# Replace with your GitHub token or use SSH
GH_TOKEN = 'YOUR_GITHUB_TOKEN'   # Settings → Developer settings → PAT
GH_USER  = 'Amitesh-Verma003'
GH_REPO  = 'gril_assessment_3'

!git config user.email 'amiteshverma003@gmail.com'
!git config user.name  'Amitesh-Verma003'

# Copy best weights to models/weights/ so they're tracked
import shutil
shutil.copy(BEST_WEIGHTS, 'models/weights/best.pt')

!git add models/weights/best.pt runs/train/ppe_{MODEL_VARIANT}/results.csv
!git commit -m "feat: add trained weights and results (ppe_{MODEL_VARIANT})"
!git push https://{GH_TOKEN}@github.com/{GH_USER}/{GH_REPO}.git main